# Obesity Level Classification Using Artificial Intelligence Techniques

**Dataset:** Estimation of Obesity Levels Based on Eating Habits and Physical Condition  
**Kaggle:** https://www.kaggle.com/datasets/fatemehmehrparvar/obesity-levels  

## 1. Import Libraries

In [1]:
import os, sys, warnings, platform
warnings.filterwarnings('ignore')

# Suppress TensorFlow logs
os.environ['TF_CPP_MIN_LOG_LEVEL']  = '3'
os.environ['CUDA_VISIBLE_DEVICES']  = '-1'   # CPU only — prevents crash

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import mode

from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.metrics         import (accuracy_score, classification_report,
                                     confusion_matrix, silhouette_score,
                                     davies_bouldin_score)
from sklearn.ensemble        import (RandomForestClassifier,
                                     GradientBoostingClassifier,
                                     VotingClassifier)
from sklearn.svm     import SVC
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import tensorflow as tf
from tensorflow              import keras
from tensorflow.keras        import layers
from tensorflow.keras.utils  import to_categorical

# Limit GPU memory growth (safe on all machines)
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

np.random.seed(42)
tf.random.set_seed(42)

print("All libraries imported successfully")
print(f"Python     : {sys.version.split()[0]}")
print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
print(f"Platform   : {platform.system()} {platform.machine()}")


All libraries imported successfully
Python     : 3.10.20
TensorFlow : 2.12.0
NumPy      : 1.23.5
Pandas     : 2.3.3
Platform   : Darwin arm64


## 2. Data Loading and Pre-Processing

### 2.1 Load Dataset
> **Download from Kaggle:** https://www.kaggle.com/datasets/fatemehmehrparvar/obesity-levels  



In [7]:
import pandas as pd

file_path = "/Users/manthan/Downloads/coursework/Applied AI/OBESITY LEVEL CLASSIFICATION USING ARTIFICIAL INTELLIGENCE TECHNIQUES /OBESITY-LEVEL-CLASSIFICATION-USING-ARTIFICIAL-INTELLIGENCE-TECHNIQUES-/Data /ObesityDataSet_raw_and_data_sinthetic.csv"

df = pd.read_csv(file_path)
print(df.head())

    Age  Gender  Height  Weight        CALC FAVC  FCVC  NCP  SCC SMOKE  CH2O  \
0  21.0  Female    1.62    64.0          no   no   2.0  3.0   no    no   2.0   
1  21.0  Female    1.52    56.0   Sometimes   no   3.0  3.0  yes   yes   3.0   
2  23.0    Male    1.80    77.0  Frequently   no   2.0  3.0   no    no   2.0   
3  27.0    Male    1.80    87.0  Frequently   no   3.0  3.0   no    no   2.0   
4  22.0    Male    1.78    89.8   Sometimes   no   2.0  1.0   no    no   2.0   

  family_history_with_overweight  FAF  TUE       CAEC                 MTRANS  \
0                            yes  0.0  1.0  Sometimes  Public_Transportation   
1                            yes  3.0  0.0  Sometimes  Public_Transportation   
2                            yes  2.0  1.0  Sometimes  Public_Transportation   
3                             no  2.0  0.0  Sometimes                Walking   
4                             no  0.0  0.0  Sometimes  Public_Transportation   

            NObeyesdad  
0        Norm

## 2.2 Exploratory Data Analysis

#### Data Types
The below chart identifies what type of data is recorded for each of these columns within the data set. For example, we will find out if each column is recorded by number (numeric), by name (categorical), or by whether the answer is true or false (boolean). 

#### Missing Values
- One way to validate the quality of the data is by checking for any missing values.
- If no missing cells are found → Data quality is acceptable.
  
#### Descriptive Statistics
- Before models are created, all existing records have to be processed first. In this section will show you some numerical columns using      statistics: -
- Mean → Average value  
- Std → Standard deviation (spread)  
- Min/Max → Range of values  
- Quartiles (25%, 50%, 75%) → Data distribution  


In [9]:
print("Data Types\n")

# Data Types
print("Data Types of Each Column:")
for col, dtype in df.dtypes.items():
    print(f"   - {col}: {dtype}")
print()

# Missing Values
print("Missing Values Check:")
missing = df.isnull().sum()

if missing.sum() == 0:
    print("   Great! No missing values found in the dataset.")
else:
    print("   Columns with missing values:")
    for col, val in missing[missing > 0].items():
        print(f"   - {col}: {val} missing values")
print()

# Descriptive Statistics
print("Summary Statistics:")
display(df.describe().round(3))

Data Types

Data Types of Each Column:
   - Age: float64
   - Gender: object
   - Height: float64
   - Weight: float64
   - CALC: object
   - FAVC: object
   - FCVC: float64
   - NCP: float64
   - SCC: object
   - SMOKE: object
   - CH2O: float64
   - family_history_with_overweight: object
   - FAF: float64
   - TUE: float64
   - CAEC: object
   - MTRANS: object
   - NObeyesdad: object

Missing Values Check:
   Great! No missing values found in the dataset.

Summary Statistics:


,Age,Height,Weight,FCVC,NCP,CH2O,FAF,TUE
count,2111.000,2111.000,2111.000,2111.000,2111.000,2111.000,2111.000,2111.000
mean,24.313,1.702,86.586,2.419,2.686,2.008,1.010,0.658
std,6.346,0.093,26.191,0.534,0.778,0.613,0.851,0.609
min,14.000,1.450,39.000,1.000,1.000,1.000,0.000,0.000
25%,19.947,1.630,65.473,2.000,2.659,1.585,0.125,0.000
50%,22.778,1.700,83.000,2.386,3.000,2.000,1.000,0.625
75%,26.000,1.768,107.431,3.000,3.000,2.477,1.667,1.000
max,61.000,1.980,173.000,3.000,4.000,3.000,3.000,2.000


## 2.3 Encode Categorical Features

#### Categorical Encoding

Categorical features were converted into numerical values using **Label Encoding**.

#### Encoded Columns
- Gender, family_history_with_overweight, FAVC  
- CAEC, SMOKE, SCC, CALC, MTRANS  

Each category is assigned a unique number (e.g., Male → 1, Female → 0).

#### Target Encoding
The target variable NObeyesdad (obesity level) was also encoded into numerical classes.

- `class_names`: Stores class labels  
- `n_classes`: Total number of classes  


In [10]:
categorical_cols = ['Gender','family_history_with_overweight','FAVC',
                    'CAEC','SMOKE','SCC','CALC','MTRANS']

le_dict    = {}
df_encoded = df.copy()
for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    le_dict[col]    = le
    print(f"  {col}: {list(le.classes_)}")

le_target                = LabelEncoder()
df_encoded['NObeyesdad'] = le_target.fit_transform(df_encoded['NObeyesdad'])
class_names              = le_target.classes_
n_classes                = len(class_names)
print(f"\nTarget classes ({n_classes}): {list(class_names)}")

  Gender: ['Female', 'Male']
  family_history_with_overweight: ['no', 'yes']
  FAVC: ['no', 'yes']
  CAEC: ['Always', 'Frequently', 'Sometimes', 'no']
  SMOKE: ['no', 'yes']
  SCC: ['no', 'yes']
  CALC: ['Always', 'Frequently', 'Sometimes', 'no']
  MTRANS: ['Automobile', 'Bike', 'Motorbike', 'Public_Transportation', 'Walking']

Target classes (7): ['Insufficient_Weight', 'Normal_Weight', 'Obesity_Type_I', 'Obesity_Type_II', 'Obesity_Type_III', 'Overweight_Level_I', 'Overweight_Level_II']
